# Transfer to WCCI (`tr_wcci`)

This notebook analyses the transfer runs downloaded from W&B into
`outputs/run_data/tr` and the deterministic 50-episode full-test evaluation
JSON files in `outputs/full_test_eval/tr_wcci_50episodes`.

The experiment compares **scratch WCCI training** against **frozen encoder
transfer** from the bus14 graph-screening checkpoints. There is currently one
seed (`s0`) per condition, so the notebook reports per-run curves and paired
frozen-minus-scratch deltas rather than seed uncertainty bands.

In [42]:
from pathlib import Path
import gzip
import json
import re
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
REPO_DIR = TASK_DIR.parent
print("task directory:", TASK_DIR)

task directory: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task


## Analysis Controls

The notebook searches local downloaded files only. If you download more
transfer runs or more full-test evaluations later, rerun all cells and the
tables/figures update automatically.

In [43]:
RUN_DATA_DIR = TASK_DIR / "outputs" / "run_data" / "tr"
RUNS_DIR = RUN_DATA_DIR / "runs"
FULL_TEST_DIR = TASK_DIR / "outputs" / "full_test_eval" / "tr_wcci_50episodes"
EXPORT_DIR = TASK_DIR / "outputs" / "tr_wcci_analysis"
FIG_DIR = EXPORT_DIR / "figures"

SAVE_FIGURES = True
SHOW_FIGURES = True
SMOOTH_WINDOW = 5
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None  # None -> common downloaded horizon
EXPECTED_FULL_TEST_EPISODES = 50

SURVIVAL_METRIC_CANDIDATES = {
    "test": ["test/charts/episodic_survival", "test/episodic_survival"],
    "train_eval": [
        "train_eval/charts/episodic_survival",
        "train_eval/episodic_survival",
    ],
}
ACTION0_METRIC_PATTERNS = {
    "test": r"^test/explain/frac_action_0_agent_\d+$",
    "train_eval": r"^train_eval/explain/frac_action_0_agent_\d+$",
    "train": r"^train/frac_action_0_agent_\d+$",
}

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("run-data folder:", RUN_DATA_DIR)
print("full-test folder:", FULL_TEST_DIR)
print("export folder:", EXPORT_DIR)

run-data folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/tr
full-test folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/full_test_eval/tr_wcci_50episodes
export folder: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis


## Helpers

Run names encode three factors:

- architecture: `s2_e0n0v0`, `s2_e1n0v0`, `hmd_gb2a_la2b`, or `hmd_gb2a_lb2a`;
- arm: `scratch` or `frozen`;
- seed: currently `s0`.

In [44]:
VARIANT_LABELS = {
    "s2_e0n0v0": "Bus e0n0v0",
    "s2_e1n0v0": "Bus e1n0v0",
    "hmd_gb2a_la2b": "HMD gb2a/la2b",
    "hmd_gb2a_lb2a": "HMD gb2a/lb2a",
}
VARIANT_ORDER = [
    "s2_e0n0v0",
    "s2_e1n0v0",
    "hmd_gb2a_la2b",
    "hmd_gb2a_lb2a",
]
ARM_LABELS = {"scratch": "scratch", "frozen": "frozen transfer"}
ARM_ORDER = ["scratch", "frozen"]


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-") or "plot"


def save_figure(fig, name):
    if fig is None or not SAVE_FIGURES:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved:", path)
    return path


def maybe_show(fig):
    if fig is not None and SHOW_FIGURES:
        fig.show()
    return fig


def load_json(path):
    with Path(path).open() as file:
        return json.load(file)


def parse_run_name(run_name):
    match = re.match(r"^tr_(?P<variant>.+)_(?P<arm>frozen|scratch)_s(?P<seed>\d+)$", str(run_name))
    if not match:
        raise ValueError(f"Unexpected transfer run name: {run_name!r}")
    variant = match.group("variant")
    arm = match.group("arm")
    seed = int(match.group("seed"))
    return {
        "variant": variant,
        "variant_label": VARIANT_LABELS.get(variant, variant),
        "arm": arm,
        "arm_label": ARM_LABELS.get(arm, arm),
        "seed": seed,
        "condition": f"{VARIANT_LABELS.get(variant, variant)} | {ARM_LABELS.get(arm, arm)}",
    }


def survival_scale(series):
    numeric = pd.to_numeric(series, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0
    return 100.0 if max_value <= 1.5 else 1.0


def first_available_column(frame, candidates):
    for column in candidates:
        if column in frame.columns and pd.to_numeric(frame[column], errors="coerce").notna().any():
            return column
    return None


def step_column(frame):
    for column in ["_step", "global_step", "step"]:
        if column in frame.columns:
            return column
    raise KeyError("No step column found in history")

## Build Run Catalog

The catalog joins W&B metadata, saved run config, and factors parsed from the
run name. This also lets us confirm which runs used frozen transferred
encoders and where each frozen run loaded its encoder from.

In [45]:
def catalog_record(run_dir):
    metadata = load_json(run_dir / "metadata.json")
    config = load_json(run_dir / "config.json")
    run_name = str(metadata.get("name") or metadata.get("exp_tag") or run_dir.name.split("__", 1)[0])
    factors = parse_run_name(run_name)
    transfer_checkpoint = str(config.get("transfer_encoder_checkpoint", "") or "")
    record = {
        "run_name": run_name,
        "run_id": metadata.get("id"),
        "exp_tag": metadata.get("exp_tag"),
        "state": metadata.get("state"),
        "group": metadata.get("group"),
        "created_at": metadata.get("created_at"),
        "run_dir": str(run_dir),
        "history_csv": str(run_dir / "history.csv.gz"),
        "history_parquet": str(run_dir / "history.parquet"),
        "summary_json": str(run_dir / "summary.json"),
        "env_id": config.get("env_id"),
        "actor_encoder": config.get("actor_encoder"),
        "gnn_type": config.get("gnn_type"),
        "gnn_graph_type": config.get("gnn_graph_type"),
        "gnn_include_neighbors": config.get("gnn_include_neighbors"),
        "gnn_add_substation_edges": config.get("gnn_add_substation_edges"),
        "gnn_add_substation_nodes": config.get("gnn_add_substation_nodes"),
        "gnn_generator_edge_direction": config.get("gnn_generator_edge_direction"),
        "gnn_load_edge_direction": config.get("gnn_load_edge_direction"),
        "gnn_line_node_edge_direction": config.get("gnn_line_node_edge_direction"),
        "total_timesteps": config.get("total_timesteps"),
        "transfer_freeze_encoder": bool(config.get("transfer_freeze_encoder", False)),
        "transfer_encoder_checkpoint": transfer_checkpoint,
        "transfer_source": Path(transfer_checkpoint).stem if transfer_checkpoint else "",
    }
    record.update(factors)
    return record


run_dirs = sorted(path for path in RUNS_DIR.iterdir() if path.is_dir()) if RUNS_DIR.exists() else []
catalog = pd.DataFrame([catalog_record(path) for path in run_dirs])
if catalog.empty:
    raise FileNotFoundError(f"No downloaded transfer runs found under {RUNS_DIR}")

catalog["variant"] = pd.Categorical(catalog["variant"], VARIANT_ORDER, ordered=True)
catalog["arm"] = pd.Categorical(catalog["arm"], ARM_ORDER, ordered=True)
catalog = catalog.sort_values(["variant", "arm", "seed"]).reset_index(drop=True)
catalog.to_csv(EXPORT_DIR / "tr_wcci_catalog.csv", index=False)

print(f"Downloaded runs: {len(catalog)}")
display(catalog[[
    "run_name",
    "variant_label",
    "arm_label",
    "seed",
    "env_id",
    "gnn_graph_type",
    "gnn_add_substation_edges",
    "gnn_generator_edge_direction",
    "gnn_load_edge_direction",
    "transfer_freeze_encoder",
    "transfer_source",
]])

Downloaded runs: 8


,run_name,variant_label,arm_label,seed,env_id,gnn_graph_type,gnn_add_substation_edges,gnn_generator_edge_direction,gnn_load_edge_direction,transfer_freeze_encoder,transfer_source
0,tr_s2_e0n0v0_scratch_s0,Bus e0n0v0,scratch,0,bus36_wcci_nomaint,bus,False,bidirectional,bidirectional,False,
1,tr_s2_e0n0v0_frozen_s0,Bus e0n0v0,frozen transfer,0,bus36_wcci_nomaint,bus,False,bidirectional,bidirectional,True,best_test_gs_s2_bus_n0_none_e0n0v0_s0
2,tr_s2_e1n0v0_scratch_s0,Bus e1n0v0,scratch,0,bus36_wcci_nomaint,bus,True,bidirectional,bidirectional,False,
3,tr_s2_e1n0v0_frozen_s0,Bus e1n0v0,frozen transfer,0,bus36_wcci_nomaint,bus,True,bidirectional,bidirectional,True,best_test_gs_s2_bus_n0_none_e1n0v0_s0
4,tr_hmd_gb2a_la2b_scratch_s0,HMD gb2a/la2b,scratch,0,bus36_wcci_nomaint,heterogeneous,False,busbar_to_asset,asset_to_busbar,False,
5,tr_hmd_gb2a_la2b_frozen_s0,HMD gb2a/la2b,frozen transfer,0,bus36_wcci_nomaint,heterogeneous,False,busbar_to_asset,asset_to_busbar,True,best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s0
6,tr_hmd_gb2a_lb2a_scratch_s0,HMD gb2a/lb2a,scratch,0,bus36_wcci_nomaint,heterogeneous,False,busbar_to_asset,busbar_to_asset,False,
7,tr_hmd_gb2a_lb2a_frozen_s0,HMD gb2a/lb2a,frozen transfer,0,bus36_wcci_nomaint,heterogeneous,False,busbar_to_asset,busbar_to_asset,True,best_test_gs_hmd_hetero_n0_none_gb2a_lb2a_s0


## Load Training Curves

Training curves are the W&B scalar histories. Test/train-eval survival during
training is a small evaluation window, not the final 50-episode statistic.

In [46]:
def read_history(row):
    csv_path = Path(row["history_csv"])
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    history = pd.read_csv(csv_path, compression="gzip", low_memory=False)
    step_col = step_column(history)
    history["_step_resolved"] = pd.to_numeric(history[step_col], errors="coerce")
    for key in [
        "run_name",
        "run_id",
        "variant",
        "variant_label",
        "arm",
        "arm_label",
        "seed",
        "condition",
    ]:
        history[key] = row[key]
    history["step_millions"] = history["_step_resolved"] / 1_000_000
    return history


histories = [read_history(row) for row in catalog.to_dict("records")]
history_wide = pd.concat(histories, ignore_index=True, sort=False)
print("history rows:", len(history_wide), "columns:", len(history_wide.columns))
print("max step by run:")
display(
    history_wide.groupby(["variant_label", "arm_label", "run_name"], observed=True)["_step_resolved"]
    .max()
    .reset_index()
    .sort_values(["variant_label", "arm_label"])
)

history rows: 2888 columns: 134
max step by run:


,variant_label,arm_label,run_name,_step_resolved
0,Bus e0n0v0,frozen transfer,tr_s2_e0n0v0_frozen_s0,14971392.0
1,Bus e0n0v0,scratch,tr_s2_e0n0v0_scratch_s0,14971392.0
2,Bus e1n0v0,frozen transfer,tr_s2_e1n0v0_frozen_s0,14971392.0
3,Bus e1n0v0,scratch,tr_s2_e1n0v0_scratch_s0,14971392.0
4,HMD gb2a/la2b,frozen transfer,tr_hmd_gb2a_la2b_frozen_s0,14971392.0
5,HMD gb2a/la2b,scratch,tr_hmd_gb2a_la2b_scratch_s0,14971392.0
6,HMD gb2a/lb2a,frozen transfer,tr_hmd_gb2a_lb2a_frozen_s0,14971392.0
7,HMD gb2a/lb2a,scratch,tr_hmd_gb2a_lb2a_scratch_s0,14971392.0


In [47]:
def survival_curve_rows(history_wide):
    frames = []
    meta_cols = [
        "run_name",
        "run_id",
        "variant",
        "variant_label",
        "arm",
        "arm_label",
        "seed",
        "condition",
        "_step_resolved",
        "step_millions",
    ]
    for run_name, run_history in history_wide.groupby("run_name", sort=False):
        for split, candidates in SURVIVAL_METRIC_CANDIDATES.items():
            metric = first_available_column(run_history, candidates)
            if metric is None:
                continue
            values = pd.to_numeric(run_history[metric], errors="coerce")
            scale = survival_scale(values)
            frame = run_history[meta_cols].copy()
            frame["split"] = split
            frame["metric"] = metric
            frame["survival_pct"] = values * scale
            frames.append(frame.dropna(subset=["_step_resolved", "survival_pct"]))
    if not frames:
        return pd.DataFrame()
    curves = pd.concat(frames, ignore_index=True, sort=False)
    curves = curves.sort_values(["run_name", "split", "_step_resolved"])
    if SMOOTH_WINDOW and SMOOTH_WINDOW > 1:
        curves["survival_pct_smooth"] = curves.groupby(["run_name", "split"], observed=True)["survival_pct"].transform(
            lambda values: values.rolling(SMOOTH_WINDOW, min_periods=1).mean()
        )
    else:
        curves["survival_pct_smooth"] = curves["survival_pct"]
    return curves


def action0_curve_rows(history_wide):
    frames = []
    meta_cols = [
        "run_name",
        "run_id",
        "variant",
        "variant_label",
        "arm",
        "arm_label",
        "seed",
        "condition",
        "_step_resolved",
        "step_millions",
    ]
    for split, pattern in ACTION0_METRIC_PATTERNS.items():
        regex = re.compile(pattern)
        columns = [column for column in history_wide.columns if regex.match(str(column))]
        if not columns:
            continue
        values = history_wide[columns].apply(pd.to_numeric, errors="coerce")
        frame = history_wide[meta_cols].copy()
        frame["split"] = split
        frame["action0_frac"] = values.mean(axis=1, skipna=True)
        frame["n_action0_agents"] = values.notna().sum(axis=1)
        frames.append(frame.dropna(subset=["_step_resolved", "action0_frac"]))
    if not frames:
        return pd.DataFrame()
    curves = pd.concat(frames, ignore_index=True, sort=False)
    curves = curves.sort_values(["run_name", "split", "_step_resolved"])
    if SMOOTH_WINDOW and SMOOTH_WINDOW > 1:
        curves["action0_frac_smooth"] = curves.groupby(["run_name", "split"], observed=True)["action0_frac"].transform(
            lambda values: values.rolling(SMOOTH_WINDOW, min_periods=1).mean()
        )
    else:
        curves["action0_frac_smooth"] = curves["action0_frac"]
    return curves


survival_curves = survival_curve_rows(history_wide)
action0_curves = action0_curve_rows(history_wide)
survival_curves.to_csv(EXPORT_DIR / "tr_wcci_survival_curves.csv", index=False)
action0_curves.to_csv(EXPORT_DIR / "tr_wcci_action0_curves.csv", index=False)

print("survival curves:", survival_curves.shape)
display(survival_curves.groupby(["split", "variant_label", "arm_label"], observed=True).agg(
    points=("survival_pct", "count"),
    max_step_m=("step_millions", "max"),
    metric=("metric", "first"),
).reset_index())

print("action-0 curves:", action0_curves.shape)
if not action0_curves.empty:
    display(action0_curves.groupby(["split", "variant_label", "arm_label"], observed=True).agg(
        points=("action0_frac", "count"),
        agents=("n_action0_agents", "max"),
    ).reset_index())

survival curves: (2880, 14)


,split,variant_label,arm_label,points,max_step_m,metric
0,test,Bus e0n0v0,frozen transfer,180,14.92992,test/charts/episodic_survival
1,test,Bus e0n0v0,scratch,180,14.92992,test/charts/episodic_survival
2,test,Bus e1n0v0,frozen transfer,180,14.92992,test/charts/episodic_survival
3,test,Bus e1n0v0,scratch,180,14.92992,test/charts/episodic_survival
4,test,HMD gb2a/la2b,frozen transfer,180,14.92992,test/charts/episodic_survival
5,test,HMD gb2a/la2b,scratch,180,14.92992,test/charts/episodic_survival
6,test,HMD gb2a/lb2a,frozen transfer,180,14.92992,test/charts/episodic_survival
7,test,HMD gb2a/lb2a,scratch,180,14.92992,test/charts/episodic_survival
8,train_eval,Bus e0n0v0,frozen transfer,180,14.92992,train_eval/charts/episodic_survival
9,train_eval,Bus e0n0v0,scratch,180,14.92992,train_eval/charts/episodic_survival


action-0 curves: (5768, 14)


,split,variant_label,arm_label,points,agents
0,test,Bus e0n0v0,frozen transfer,180,4
1,test,Bus e0n0v0,scratch,180,4
2,test,Bus e1n0v0,frozen transfer,180,4
3,test,Bus e1n0v0,scratch,180,4
4,test,HMD gb2a/la2b,frozen transfer,180,4
5,test,HMD gb2a/la2b,scratch,180,4
6,test,HMD gb2a/lb2a,frozen transfer,180,4
7,test,HMD gb2a/lb2a,scratch,180,4
8,train,Bus e0n0v0,frozen transfer,361,4
9,train,Bus e0n0v0,scratch,361,4


## Load 50-Episode Full-Test Evaluations

These JSON files are the deterministic held-out evaluations over 50 chronics.
They use the selected best checkpoint for each run, so the checkpoint step can
differ from the final training step.

In [48]:
def full_test_record(path):
    data = load_json(path)
    run_name = path.stem
    factors = parse_run_name(run_name)
    checkpoint = str(data.get("checkpoint", ""))
    return {
        "run_name": run_name,
        "full_test_json": str(path),
        "checkpoint": checkpoint,
        "checkpoint_name": Path(checkpoint).name,
        "checkpoint_global_step": int(data.get("checkpoint_global_step", 0) or 0),
        "checkpoint_step_millions": float(data.get("checkpoint_global_step", 0) or 0) / 1_000_000,
        "split": data.get("split"),
        "eval_episodes": int(data.get("eval_episodes", 0) or 0),
        "survival_frac": float(data.get("survival_frac", np.nan)),
        "survival_percent": float(data.get("survival_percent", np.nan)),
        "deterministic_eval": bool(data.get("deterministic_eval", False)),
        "eval_action_heuristic": data.get("eval_action_heuristic"),
        "obs_normalization": data.get("obs_normalization"),
        "norm_obs_effective": data.get("norm_obs_effective"),
        "obs_stats_available": data.get("obs_stats_available"),
        **factors,
    }


full_test_paths = sorted(FULL_TEST_DIR.glob("tr_*.json")) if FULL_TEST_DIR.exists() else []
full_test = pd.DataFrame([full_test_record(path) for path in full_test_paths])
if full_test.empty:
    warnings.warn(f"No full-test JSON files found in {FULL_TEST_DIR}")
else:
    full_test["variant"] = pd.Categorical(full_test["variant"], VARIANT_ORDER, ordered=True)
    full_test["arm"] = pd.Categorical(full_test["arm"], ARM_ORDER, ordered=True)
    full_test = full_test.sort_values(["variant", "arm", "seed"]).reset_index(drop=True)
    full_test.to_csv(EXPORT_DIR / "tr_wcci_full_test_50episodes.csv", index=False)
    print(f"Full-test files: {len(full_test)}")
    display(full_test[[
        "run_name",
        "variant_label",
        "arm_label",
        "checkpoint_step_millions",
        "eval_episodes",
        "survival_percent",
        "obs_normalization",
        "eval_action_heuristic",
    ]])
    if EXPECTED_FULL_TEST_EPISODES is not None:
        bad = full_test[full_test["eval_episodes"] != EXPECTED_FULL_TEST_EPISODES]
        if not bad.empty:
            warnings.warn(
                "Some full-test results do not have "
                f"{EXPECTED_FULL_TEST_EPISODES} episodes: "
                + ", ".join(bad["run_name"].tolist())
            )

Full-test files: 8


,run_name,variant_label,arm_label,checkpoint_step_millions,eval_episodes,survival_percent,obs_normalization,eval_action_heuristic
0,tr_s2_e0n0v0_scratch_s0,Bus e0n0v0,scratch,7.382016,50,47.617713,checkpoint_stats,none
1,tr_s2_e0n0v0_frozen_s0,Bus e0n0v0,frozen transfer,0.580608,50,30.482759,checkpoint_stats,none
2,tr_s2_e1n0v0_scratch_s0,Bus e1n0v0,scratch,4.313088,50,51.311585,checkpoint_stats,none
3,tr_s2_e1n0v0_frozen_s0,Bus e1n0v0,frozen transfer,13.022208,50,13.944431,checkpoint_stats,none
4,tr_hmd_gb2a_la2b_scratch_s0,HMD gb2a/la2b,scratch,7.382016,50,52.320764,checkpoint_stats,none
5,tr_hmd_gb2a_la2b_frozen_s0,HMD gb2a/la2b,frozen transfer,14.681088,50,52.595386,checkpoint_stats,none
6,tr_hmd_gb2a_lb2a_scratch_s0,HMD gb2a/lb2a,scratch,7.382016,50,54.457703,checkpoint_stats,none
7,tr_hmd_gb2a_lb2a_frozen_s0,HMD gb2a/lb2a,frozen transfer,10.119168,50,54.370131,checkpoint_stats,none


## Common-Horizon Training Summary

To compare curves fairly, this section picks the last downloaded test-eval
point available for every run. With the current download this should be near
15M steps.

In [49]:
def last_at_or_before(frame, step):
    eligible = frame[frame["_step_resolved"] <= step].sort_values("_step_resolved")
    if eligible.empty:
        return None
    return eligible.iloc[-1]


test_curves = survival_curves[survival_curves["split"].eq("test")].copy()
max_steps = test_curves.groupby("run_name")["_step_resolved"].max()
if COMPARISON_BUDGET_STEPS is None:
    comparison_step = int(max_steps.min()) if not max_steps.empty else TARGET_BUDGET_STEPS
else:
    comparison_step = int(COMPARISON_BUDGET_STEPS)
print(f"Comparison step: {comparison_step:,} ({comparison_step / 1_000_000:.2f}M)")

rows = []
for run_name, run_curves in test_curves.groupby("run_name", sort=False):
    row = last_at_or_before(run_curves, comparison_step)
    if row is None:
        continue
    rows.append(row.to_dict())
common_summary = pd.DataFrame(rows)
if not common_summary.empty:
    common_summary = common_summary.merge(
        catalog[["run_name", "transfer_source", "env_id", "total_timesteps"]],
        on="run_name",
        how="left",
    )
    common_summary = common_summary.sort_values(["variant", "arm", "seed"]).reset_index(drop=True)
    common_summary.to_csv(EXPORT_DIR / "tr_wcci_training_common_horizon_summary.csv", index=False)
    display(common_summary[[
        "run_name",
        "variant_label",
        "arm_label",
        "step_millions",
        "survival_pct",
        "survival_pct_smooth",
        "transfer_source",
    ]])

Comparison step: 14,929,920 (14.93M)


,run_name,variant_label,arm_label,step_millions,survival_pct,survival_pct_smooth,transfer_source
0,tr_hmd_gb2a_la2b_frozen_s0,HMD gb2a/la2b,frozen transfer,14.92992,42.506822,46.931531,best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s0
1,tr_hmd_gb2a_la2b_scratch_s0,HMD gb2a/la2b,scratch,14.92992,43.559911,49.039444,
2,tr_hmd_gb2a_lb2a_frozen_s0,HMD gb2a/lb2a,frozen transfer,14.92992,44.114364,50.812702,best_test_gs_hmd_hetero_n0_none_gb2a_lb2a_s0
3,tr_hmd_gb2a_lb2a_scratch_s0,HMD gb2a/lb2a,scratch,14.92992,43.896056,50.749690,
4,tr_s2_e0n0v0_frozen_s0,Bus e0n0v0,frozen transfer,14.92992,8.177871,10.231704,best_test_gs_s2_bus_n0_none_e0n0v0_s0
5,tr_s2_e0n0v0_scratch_s0,Bus e0n0v0,scratch,14.92992,13.564872,13.848425,
6,tr_s2_e1n0v0_frozen_s0,Bus e1n0v0,frozen transfer,14.92992,19.636567,18.243860,best_test_gs_s2_bus_n0_none_e1n0v0_s0
7,tr_s2_e1n0v0_scratch_s0,Bus e1n0v0,scratch,14.92992,17.060283,15.238650,


In [50]:
def paired_delta_table(value_frame, value_col, step_col=None, label="value"):
    if value_frame.empty:
        return pd.DataFrame()
    index_cols = ["variant", "variant_label", "seed"]
    wide = value_frame.pivot_table(
        index=index_cols,
        columns="arm",
        values=value_col,
        aggfunc="first",
        observed=True,
    ).reset_index()
    if "frozen" in wide.columns and "scratch" in wide.columns:
        wide[f"delta_frozen_minus_scratch_{label}"] = wide["frozen"] - wide["scratch"]
    if step_col is not None and step_col in value_frame.columns:
        step_wide = value_frame.pivot_table(
            index=index_cols,
            columns="arm",
            values=step_col,
            aggfunc="first",
            observed=True,
        ).reset_index()
        step_wide = step_wide.rename(columns={
            "frozen": "frozen_checkpoint_step_m",
            "scratch": "scratch_checkpoint_step_m",
        })
        wide = wide.merge(step_wide, on=index_cols, how="left")
    return wide.sort_values(["variant", "seed"]).reset_index(drop=True)


train_delta = paired_delta_table(
    common_summary,
    "survival_pct_smooth",
    label="training_test_survival_pct",
)
full_test_delta = paired_delta_table(
    full_test,
    "survival_percent",
    step_col="checkpoint_step_millions",
    label="full_test_survival_pct",
)

train_delta.to_csv(EXPORT_DIR / "tr_wcci_training_delta_frozen_minus_scratch.csv", index=False)
full_test_delta.to_csv(EXPORT_DIR / "tr_wcci_full_test_delta_frozen_minus_scratch.csv", index=False)

print("Training test-eval frozen-minus-scratch delta at common horizon:")
display(train_delta)
print("50-episode full-test frozen-minus-scratch delta:")
display(full_test_delta)

Training test-eval frozen-minus-scratch delta at common horizon:


arm,variant,variant_label,seed,frozen,scratch,delta_frozen_minus_scratch_training_test_survival_pct
0,hmd_gb2a_la2b,HMD gb2a/la2b,0,46.931531,49.039444,-2.107914
1,hmd_gb2a_lb2a,HMD gb2a/lb2a,0,50.812702,50.749690,0.063012
2,s2_e0n0v0,Bus e0n0v0,0,10.231704,13.848425,-3.616720
3,s2_e1n0v0,Bus e1n0v0,0,18.243860,15.238650,3.005210


50-episode full-test frozen-minus-scratch delta:


arm,variant,variant_label,seed,scratch,frozen,delta_frozen_minus_scratch_full_test_survival_pct,scratch_checkpoint_step_m,frozen_checkpoint_step_m
0,s2_e0n0v0,Bus e0n0v0,0,47.617713,30.482759,-17.134954,7.382016,0.580608
1,s2_e1n0v0,Bus e1n0v0,0,51.311585,13.944431,-37.367155,4.313088,13.022208
2,hmd_gb2a_la2b,HMD gb2a/la2b,0,52.320764,52.595386,0.274622,7.382016,14.681088
3,hmd_gb2a_lb2a,HMD gb2a/lb2a,0,54.457703,54.370131,-0.087571,7.382016,10.119168


## Plots

In [51]:
COLOR_MAP = {
    "s2_e0n0v0": "#1f77b4",
    "s2_e1n0v0": "#2ca02c",
    "hmd_gb2a_la2b": "#ff7f0e",
    "hmd_gb2a_lb2a": "#d62728",
}
DASH_MAP = {"scratch": "solid", "frozen": "dash"}


def plot_survival_curves(split="test", include_full_test_markers=True):
    data = survival_curves[survival_curves["split"].eq(split)].copy()
    if data.empty:
        print(f"No survival curve data for split={split!r}")
        return None
    fig = go.Figure()
    for _, meta in catalog.sort_values(["variant", "arm", "seed"]).iterrows():
        run_data = data[data["run_name"].eq(meta["run_name"])].sort_values("step_millions")
        if run_data.empty:
            continue
        variant = str(meta["variant"])
        arm = str(meta["arm"])
        fig.add_trace(go.Scatter(
            x=run_data["step_millions"],
            y=run_data["survival_pct_smooth"],
            mode="lines",
            name=meta["condition"],
            legendgroup=meta["condition"],
            line={"color": COLOR_MAP.get(variant), "dash": DASH_MAP.get(arm, "solid"), "width": 3},
            customdata=np.stack([
                run_data["survival_pct"],
                run_data["metric"].astype(str),
            ], axis=-1),
            hovertemplate=(
                f"<b>{meta['condition']}</b><br>"
                "step=%{x:.2f}M<br>"
                "smoothed survival=%{y:.2f}%<br>"
                "raw survival=%{customdata[0]:.2f}%<br>"
                "metric=%{customdata[1]}<extra></extra>"
            ),
        ))
    if include_full_test_markers and split == "test" and not full_test.empty:
        for _, row in full_test.iterrows():
            variant = str(row["variant"])
            arm = str(row["arm"])
            fig.add_trace(go.Scatter(
                x=[row["checkpoint_step_millions"]],
                y=[row["survival_percent"]],
                mode="markers",
                name=f"50-ep {row['condition']}",
                legendgroup=row["condition"],
                showlegend=False,
                marker={
                    "symbol": "star" if arm == "frozen" else "circle",
                    "size": 12,
                    "color": COLOR_MAP.get(variant),
                    "line": {"color": "black", "width": 1},
                },
                hovertemplate=(
                    f"<b>50-episode full test</b><br>{row['condition']}<br>"
                    "checkpoint step=%{x:.2f}M<br>"
                    "survival=%{y:.2f}%<extra></extra>"
                ),
            ))
    fig.update_layout(
        title=f"TR WCCI {split} survival curves",
        xaxis_title="training step (millions)",
        yaxis_title="episodic survival (%)",
        template="plotly_white",
        hovermode="x unified",
        width=1050,
        height=620,
        legend_title="condition",
    )
    fig.update_yaxes(range=[0, max(60, data["survival_pct_smooth"].max() * 1.08)])
    save_figure(fig, f"tr_wcci_{split}_survival_curves")
    return maybe_show(fig)


plot_survival_curves("test")
plot_survival_curves("train_eval", include_full_test_markers=False)

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_test_survival_curves.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_train_eval_survival_curves.html


## Frozen vs scratch pairwise survival

Each panel compares the frozen-transfer run against the train-from-scratch run for one run type. The dashed horizontal line is the do-nothing survival reference at 57.2%.


In [56]:
DO_NOTHING_SURVIVAL_PCT = 57.2

ARM_COLORS = {"scratch": "#4C78A8", "frozen": "#F58518"}
ARM_DASHES = {"scratch": "solid", "frozen": "dash"}
DO_NOTHING_COLOR = "#5F6368"


def plot_frozen_vs_scratch_survival_subplots(split="test", do_nothing_survival_pct=DO_NOTHING_SURVIVAL_PCT):
    data = survival_curves[survival_curves["split"].eq(split)].copy()
    if data.empty:
        print(f"No survival curve data for split={split!r}")
        return None

    x_min = float(data["step_millions"].min(skipna=True))
    x_max = float(data["step_millions"].max(skipna=True))

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
        subplot_titles=[VARIANT_LABELS.get(variant, variant) for variant in VARIANT_ORDER],
    )

    for index, variant in enumerate(VARIANT_ORDER):
        row = index // 2 + 1
        col = index % 2 + 1

        fig.add_trace(
            go.Scatter(
                x=[x_min, x_max],
                y=[do_nothing_survival_pct, do_nothing_survival_pct],
                mode="lines",
                name="do nothing 57.2%",
                legendgroup="do_nothing",
                showlegend=(index == 0),
                line={"color": DO_NOTHING_COLOR, "dash": "dash", "width": 1.5},
                hovertemplate="do nothing survival=%{y:.1f}%<extra></extra>",
            ),
            row=row,
            col=col,
        )

        for arm in ARM_ORDER:
            meta_rows = catalog[
                catalog["variant"].astype(str).eq(variant)
                & catalog["arm"].astype(str).eq(arm)
            ].sort_values("seed")
            for _, meta in meta_rows.iterrows():
                run_data = data[data["run_name"].eq(meta["run_name"])].sort_values("step_millions")
                if run_data.empty:
                    continue
                fig.add_trace(
                    go.Scatter(
                        x=run_data["step_millions"],
                        y=run_data["survival_pct_smooth"],
                        mode="lines",
                        name=ARM_LABELS.get(arm, arm),
                        legendgroup=arm,
                        showlegend=(index == 0),
                        line={
                            "color": ARM_COLORS.get(arm),
                            "dash": ARM_DASHES.get(arm, "solid"),
                            "width": 3,
                        },
                        customdata=np.stack([
                            run_data["survival_pct"],
                            run_data["metric"].astype(str),
                            np.repeat(str(meta["run_name"]), len(run_data)),
                        ], axis=-1),
                        hovertemplate=(
                            f"<b>{VARIANT_LABELS.get(variant, variant)}</b><br>"
                            f"{ARM_LABELS.get(arm, arm)}<br>"
                            "run=%{customdata[2]}<br>"
                            "step=%{x:.2f}M<br>"
                            "smoothed survival=%{y:.2f}%<br>"
                            "raw survival=%{customdata[0]:.2f}%<br>"
                            "metric=%{customdata[1]}<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=col,
                )

    y_max = float(data["survival_pct_smooth"].max(skipna=True) or 0.0)
    y_max = max(y_max, float(do_nothing_survival_pct))
    fig.update_yaxes(range=[0, max(60, y_max * 1.10)])
    fig.update_xaxes(title_text="step (M)", row=2, col=1)
    fig.update_xaxes(title_text="step (M)", row=2, col=2)
    fig.update_yaxes(title_text="survival (%)", row=1, col=1)
    fig.update_yaxes(title_text="survival (%)", row=2, col=1)
    fig.update_layout(
        title=f"TR WCCI frozen transfer vs train from scratch ({split} survival)",
        template="plotly_white",
        width=1050,
        height=720,
        hovermode="closest",
        legend_title="training arm",
    )
    save_figure(fig, f"tr_wcci_frozen_vs_scratch_{split}_survival_2x2")
    return maybe_show(fig)


plot_frozen_vs_scratch_survival_subplots("test")
print("done")

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_frozen_vs_scratch_test_survival_2x2.html


done


In [53]:
DO_NOTHING_FULL_TEST_SURVIVAL_PCT = 56.0


def plot_full_test_bars(do_nothing_survival_pct=DO_NOTHING_FULL_TEST_SURVIVAL_PCT):
    if full_test.empty:
        print("No full-test data")
        return None
    data = full_test.copy()
    data["variant_label"] = pd.Categorical(
        data["variant_label"],
        [VARIANT_LABELS[v] for v in VARIANT_ORDER],
        ordered=True,
    )
    fig = px.bar(
        data,
        x="variant_label",
        y="survival_percent",
        color="arm_label",
        barmode="group",
        text="survival_percent",
        category_orders={"arm_label": [ARM_LABELS[a] for a in ARM_ORDER]},
        labels={
            "variant_label": "architecture",
            "survival_percent": "50-episode survival (%)",
            "arm_label": "training arm",
        },
        hover_data=["checkpoint_step_millions", "eval_episodes", "checkpoint_name"],
        title="TR WCCI 50-episode full-test survival",
    )
    fig.add_hline(
        y=do_nothing_survival_pct,
        line_width=1.7,
        line_dash="dash",
        line_color="#5F6368",
        annotation_text="do nothing 56%",
        annotation_position="top left",
        annotation_font={"size": 12, "color": "#5F6368"},
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(template="plotly_white", width=950, height=520)
    y_max = max(float(data["survival_percent"].max(skipna=True) or 0.0), do_nothing_survival_pct)
    fig.update_yaxes(range=[0, max(60, y_max * 1.18)])
    save_figure(fig, "tr_wcci_full_test_50episodes_bars")
    return maybe_show(fig)


def plot_delta_bars(delta_frame, delta_col, title, save_name):
    if delta_frame.empty or delta_col not in delta_frame.columns:
        print("No delta data")
        return None
    data = delta_frame.copy()
    data["variant_label"] = pd.Categorical(
        data["variant_label"],
        [VARIANT_LABELS[v] for v in VARIANT_ORDER],
        ordered=True,
    )
    fig = px.bar(
        data,
        x="variant_label",
        y=delta_col,
        text=delta_col,
        labels={"variant_label": "architecture", delta_col: "frozen - scratch (percentage points)"},
        title=title,
    )
    fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="black")
    fig.update_traces(texttemplate="%{text:+.2f}", textposition="outside")
    fig.update_layout(template="plotly_white", width=900, height=470)
    max_abs = max(5, float(data[delta_col].abs().max(skipna=True) or 0) * 1.25)
    fig.update_yaxes(range=[-max_abs, max_abs])
    save_figure(fig, save_name)
    return maybe_show(fig)


plot_full_test_bars()
plot_delta_bars(
    train_delta,
    "delta_frozen_minus_scratch_training_test_survival_pct",
    "Frozen transfer effect at common W&B test-eval horizon",
    "tr_wcci_training_delta_frozen_minus_scratch",
)
plot_delta_bars(
    full_test_delta,
    "delta_frozen_minus_scratch_full_test_survival_pct",
    "Frozen transfer effect on 50-episode full test",
    "tr_wcci_full_test_delta_frozen_minus_scratch",
)

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_full_test_50episodes_bars.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_training_delta_frozen_minus_scratch.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_full_test_delta_frozen_minus_scratch.html


In [54]:
def plot_checkpoint_step_scatter():
    if full_test.empty:
        print("No full-test data")
        return None
    fig = px.scatter(
        full_test,
        x="checkpoint_step_millions",
        y="survival_percent",
        color="variant_label",
        symbol="arm_label",
        text="arm",
        labels={
            "checkpoint_step_millions": "selected checkpoint step (millions)",
            "survival_percent": "50-episode survival (%)",
            "variant_label": "architecture",
            "arm_label": "training arm",
        },
        hover_data=["run_name", "checkpoint_name", "eval_episodes"],
        title="Best-checkpoint step versus 50-episode survival",
    )
    fig.update_traces(textposition="top center", marker={"size": 11})
    fig.update_layout(template="plotly_white", width=950, height=540)
    save_figure(fig, "tr_wcci_checkpoint_step_vs_full_test_survival")
    return maybe_show(fig)


def plot_action0_curves(split="test"):
    if action0_curves.empty:
        print("No action-0 curve data")
        return None
    data = action0_curves[action0_curves["split"].eq(split)].copy()
    if data.empty:
        print(f"No action-0 data for split={split!r}")
        return None
    fig = go.Figure()
    for _, meta in catalog.sort_values(["variant", "arm", "seed"]).iterrows():
        run_data = data[data["run_name"].eq(meta["run_name"])].sort_values("step_millions")
        if run_data.empty:
            continue
        variant = str(meta["variant"])
        arm = str(meta["arm"])
        fig.add_trace(go.Scatter(
            x=run_data["step_millions"],
            y=run_data["action0_frac_smooth"],
            mode="lines",
            name=meta["condition"],
            line={"color": COLOR_MAP.get(variant), "dash": DASH_MAP.get(arm, "solid"), "width": 3},
            customdata=np.stack([run_data["n_action0_agents"]], axis=-1),
            hovertemplate=(
                f"<b>{meta['condition']}</b><br>"
                "step=%{x:.2f}M<br>"
                "mean action-0 fraction=%{y:.4f}<br>"
                "agents=%{customdata[0]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=f"TR WCCI {split} action-0 fraction",
        xaxis_title="training step (millions)",
        yaxis_title="mean action-0 fraction across agents",
        template="plotly_white",
        hovermode="x unified",
        width=1050,
        height=560,
    )
    save_figure(fig, f"tr_wcci_{split}_action0_curves")
    return maybe_show(fig)


plot_checkpoint_step_scatter()
plot_action0_curves("test")
plot_action0_curves("train_eval")

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_checkpoint_step_vs_full_test_survival.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_test_action0_curves.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis/figures/tr_wcci_train_eval_action0_curves.html


## Compact Ranking Table

This final table combines common-horizon W&B test-eval survival with the
50-episode full-test result. Sort by whichever column you trust for the
specific question: curve dynamics or held-out evaluation.

In [55]:
ranking = catalog[[
    "run_name",
    "variant_label",
    "arm_label",
    "seed",
    "transfer_source",
]].copy()
if not common_summary.empty:
    ranking = ranking.merge(
        common_summary[["run_name", "step_millions", "survival_pct_smooth"]].rename(columns={
            "step_millions": "curve_step_m",
            "survival_pct_smooth": "curve_test_survival_pct",
        }),
        on="run_name",
        how="left",
    )
if not full_test.empty:
    ranking = ranking.merge(
        full_test[[
            "run_name",
            "checkpoint_step_millions",
            "eval_episodes",
            "survival_percent",
        ]].rename(columns={"survival_percent": "full_test_survival_pct"}),
        on="run_name",
        how="left",
    )
ranking = ranking.sort_values("full_test_survival_pct", ascending=False, na_position="last")
ranking.to_csv(EXPORT_DIR / "tr_wcci_compact_ranking.csv", index=False)
display(ranking)
print("Saved CSV/HTML outputs under:", EXPORT_DIR)

,run_name,variant_label,arm_label,seed,transfer_source,curve_step_m,curve_test_survival_pct,checkpoint_step_millions,eval_episodes,full_test_survival_pct
6,tr_hmd_gb2a_lb2a_scratch_s0,HMD gb2a/lb2a,scratch,0,,14.92992,50.749690,7.382016,50,54.457703
7,tr_hmd_gb2a_lb2a_frozen_s0,HMD gb2a/lb2a,frozen transfer,0,best_test_gs_hmd_hetero_n0_none_gb2a_lb2a_s0,14.92992,50.812702,10.119168,50,54.370131
5,tr_hmd_gb2a_la2b_frozen_s0,HMD gb2a/la2b,frozen transfer,0,best_test_gs_hmd_hetero_n0_none_gb2a_la2b_s0,14.92992,46.931531,14.681088,50,52.595386
4,tr_hmd_gb2a_la2b_scratch_s0,HMD gb2a/la2b,scratch,0,,14.92992,49.039444,7.382016,50,52.320764
2,tr_s2_e1n0v0_scratch_s0,Bus e1n0v0,scratch,0,,14.92992,15.238650,4.313088,50,51.311585
0,tr_s2_e0n0v0_scratch_s0,Bus e0n0v0,scratch,0,,14.92992,13.848425,7.382016,50,47.617713
1,tr_s2_e0n0v0_frozen_s0,Bus e0n0v0,frozen transfer,0,best_test_gs_s2_bus_n0_none_e0n0v0_s0,14.92992,10.231704,0.580608,50,30.482759
3,tr_s2_e1n0v0_frozen_s0,Bus e1n0v0,frozen transfer,0,best_test_gs_s2_bus_n0_none_e1n0v0_s0,14.92992,18.243860,13.022208,50,13.944431


Saved CSV/HTML outputs under: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/tr_wcci_analysis
